<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-02-dl-programmer/01_nn_fundamentals.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-02-dl-programmer/01_nn_fundamentals.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# Neural Network Fundamentals: From Scratch to Autograd

**Module 4 - Notebook 1** | Skillsoft Track 2: DL Programmer

## Objectives
- Implement MLP from scratch (NumPy)
- Verify backprop with PyTorch autograd
- Understand activation functions, initialization
- Universal approximation theorem demo

In [ ]:
# Setup
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)

print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

## 1. NumPy MLP from Scratch

In [ ]:
class ScratchMLP:
    """Multi-layer perceptron implemented in pure NumPy."""

    def __init__(self, layer_sizes, activations):
        self.layer_sizes = layer_sizes
        self.activations = activations
        self.task = "regression" if layer_sizes[-1] == 1 else "classification"
        self.params = {}
        self.cache = {}

        for i in range(len(layer_sizes) - 1):
            fan_in, fan_out = layer_sizes[i], layer_sizes[i + 1]
            scale = np.sqrt(2.0 / fan_in)
            self.params[f"W{i}"] = np.random.randn(fan_in, fan_out) * scale
            self.params[f"b{i}"] = np.zeros((1, fan_out))

    def _activation(self, x, name):
        if name == "relu":
            return np.maximum(0, x)
        elif name == "sigmoid":
            return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        elif name == "tanh":
            return np.tanh(x)
        elif name == "linear":
            return x
        elif name == "softmax":
            exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
            return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    def _activation_grad(self, x, name):
        if name == "relu":
            return (x > 0).astype(float)
        elif name == "sigmoid":
            s = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
            return s * (1 - s)
        elif name == "tanh":
            return 1 - np.tanh(x) ** 2
        elif name == "linear":
            return np.ones_like(x)
        elif name == "softmax":
            return x

    def forward(self, X):
        self.cache["A0"] = X
        A = X
        for i in range(len(self.layer_sizes) - 1):
            Z = A @ self.params[f"W{i}"] + self.params[f"b{i}"]
            A = self._activation(Z, self.activations[i])
            self.cache[f"Z{i+1}"] = Z
            self.cache[f"A{i+1}"] = A
        return A

    def compute_loss(self, Y_hat, Y):
        m = Y.shape[0]
        if self.task == "regression":
            Y = np.asarray(Y).reshape(Y_hat.shape)
            return float(np.mean((Y_hat - Y) ** 2))
        Y = np.asarray(Y).astype(int)
        return -np.sum(np.log(Y_hat[np.arange(m), Y] + 1e-8)) / m

    def backward(self, Y):
        m = Y.shape[0]
        grads = {}
        A_last = self.cache[f"A{len(self.layer_sizes)-1}"]
        if self.task == "regression":
            dZ = 2.0 * (A_last - np.asarray(Y).reshape(A_last.shape)) / m
        else:
            Y = np.asarray(Y).astype(int)
            dZ = A_last.copy()
            dZ[np.arange(m), Y] -= 1
            dZ /= m

        for i in reversed(range(len(self.layer_sizes) - 1)):
            A_prev = self.cache[f"A{i}"]
            grads[f"dW{i}"] = A_prev.T @ dZ
            grads[f"db{i}"] = np.sum(dZ, axis=0, keepdims=True)
            if i > 0:
                dA = dZ @ self.params[f"W{i}"].T
                dZ = dA * self._activation_grad(self.cache[f"Z{i}"], self.activations[i - 1])
        return grads

    def update(self, grads, lr):
        for key in self.params:
            self.params[key] -= lr * grads[f"d{key}"]

    def train_step(self, X, Y, lr):
        Y_hat = self.forward(X)
        loss = self.compute_loss(Y_hat, Y)
        grads = self.backward(Y)
        self.update(grads, lr)
        return loss, Y_hat

In [ ]:
# Test on synthetic data
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=2000, n_features=20, n_classes=3, n_informative=15, random_state=42
)
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = ScratchMLP([20, 64, 32, 3], ["relu", "relu", "softmax"])

losses = []
for epoch in range(100):
    idx = np.random.choice(len(X_train), 64)
    loss, _ = model.train_step(X_train[idx], y_train[idx], lr=0.01)
    losses.append(loss)
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Loss = {loss:.4f}")

# Evaluate
train_acc = np.mean(np.argmax(model.forward(X_train), axis=1) == y_train)
test_acc = np.mean(np.argmax(model.forward(X_test), axis=1) == y_test)
print(f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

## 2. Gradient Verification with PyTorch Autograd

In [ ]:
# Compare NumPy gradients with PyTorch autograd
torch_model = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 3),
)

# Copy weights from NumPy model
with torch.no_grad():
    torch_model[0].weight.copy_(torch.tensor(model.params["W0"].T, dtype=torch.float32))
    torch_model[0].bias.copy_(torch.tensor(model.params["b0"].flatten(), dtype=torch.float32))
    torch_model[2].weight.copy_(torch.tensor(model.params["W1"].T, dtype=torch.float32))
    torch_model[2].bias.copy_(torch.tensor(model.params["b1"].flatten(), dtype=torch.float32))
    torch_model[4].weight.copy_(torch.tensor(model.params["W2"].T, dtype=torch.float32))
    torch_model[4].bias.copy_(torch.tensor(model.params["b2"].flatten(), dtype=torch.float32))

# PyTorch forward + backward
X_t = torch.tensor(X_train[:64], dtype=torch.float32)
y_t = torch.tensor(y_train[:64], dtype=torch.long)

logits = torch_model(X_t)
loss_t = F.cross_entropy(logits, y_t)
loss_t.backward()

# Compare gradients
for i, (name, param) in enumerate(torch_model.named_parameters()):
    if "weight" in name:
        layer_idx = i // 2
        np_grad = model.backward(y_train[:64])[f"dW{layer_idx}"].T
        torch_grad = param.grad.numpy()
        diff = np.abs(np_grad - torch_grad).max()
        print(f"Layer {layer_idx} weight grad max diff: {diff:.6f}")
    elif "bias" in name:
        layer_idx = i // 2
        np_grad = model.backward(y_train[:64])[f"db{layer_idx}"].flatten()
        torch_grad = param.grad.numpy()
        diff = np.abs(np_grad - torch_grad).max()
        print(f"Layer {layer_idx} bias grad max diff: {diff:.6f}")

## 3. Activation Functions Comparison

In [ ]:
# Visualize activations and gradients
x = np.linspace(-5, 5, 1000)

activations = {
    "ReLU": (lambda x: np.maximum(0, x), lambda x: (x > 0).astype(float)),
    "Sigmoid": (
        lambda x: 1 / (1 + np.exp(-x)),
        lambda x: (1 / (1 + np.exp(-x))) * (1 - 1 / (1 + np.exp(-x))),
    ),
    "Tanh": (lambda x: np.tanh(x), lambda x: 1 - np.tanh(x) ** 2),
    "GELU": (
        lambda x: 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3))),
        lambda x: 0.5 * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))
        + 0.5
        * x
        * (1 / np.cosh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)) ** 2)
        * np.sqrt(2 / np.pi)
        * (1 + 3 * 0.044715 * x**2),
    ),
    "Swish": (
        lambda x: x / (1 + np.exp(-x)),
        lambda x: 1 / (1 + np.exp(-x)) + x * np.exp(-x) / (1 + np.exp(-x)) ** 2,
    ),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (name, (fn, grad_fn)) in enumerate(activations.items()):
    y = fn(x)
    gy = grad_fn(x)
    axes[idx].plot(x, y, label=name, linewidth=2)
    axes[idx].plot(x, gy, label=f"{name} grad", linestyle="--", alpha=0.7)
    axes[idx].axhline(y=0, color="k", linewidth=0.5)
    axes[idx].axvline(x=0, color="k", linewidth=0.5)
    axes[idx].set_title(name)
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 4. Weight Initialization Comparison

In [ ]:
# Compare initialization methods
def get_init_fn(name, fan_in, fan_out):
    if name == "Xavier/Glorot":
        return np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / (fan_in + fan_out))
    elif name == "He (ReLU)":
        return np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_in)
    elif name == "LeCun (SELU)":
        return np.random.randn(fan_in, fan_out) * np.sqrt(1.0 / fan_in)
    elif name == "Standard Normal":
        return np.random.randn(fan_in, fan_out)
    elif name == "Uniform [-1,1]":
        return np.random.uniform(-1, 1, (fan_in, fan_out))


init_methods = ["Xavier/Glorot", "He (ReLU)", "LeCun (SELU)", "Standard Normal", "Uniform [-1,1]"]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for idx, method in enumerate(init_methods):
    weights = get_init_fn(method, 512, 512)
    axes[idx].hist(weights.flatten(), bins=50, alpha=0.7, edgecolor="black")
    axes[idx].set_title(f"{method}\nstd={weights.std():.4f}")
    axes[idx].axvline(x=0, color="r", linestyle="--")
    axes[idx].set_xlim(-3, 3)

plt.tight_layout()
plt.show()

## 5. Universal Approximation Demo

In [ ]:
# Universal Approximation Theorem: 1 hidden layer can approximate any continuous function
def target_function(x):
    return np.sin(2 * np.pi * x) + 0.5 * np.cos(4 * np.pi * x)


x_train = np.linspace(-1, 1, 100).reshape(-1, 1)
y_train = target_function(x_train)

x_test = np.linspace(-1, 1, 500).reshape(-1, 1)
y_test = target_function(x_test)

# Train networks with different hidden sizes
hidden_sizes = [5, 10, 20, 50, 100]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, hidden in enumerate(hidden_sizes):
    model = ScratchMLP([1, hidden, 1], ["tanh", "linear"])

    losses = []
    for epoch in range(2000):
        loss, _ = model.train_step(x_train, y_train, lr=0.01)
        losses.append(loss)

    y_pred = model.forward(x_test)

    axes[idx].plot(x_test, y_test, "k-", label="True", linewidth=2)
    axes[idx].plot(x_test, y_pred, "r--", label=f"Pred (hidden={hidden})", linewidth=2)
    axes[idx].scatter(x_train, y_train, alpha=0.3, s=10)
    axes[idx].set_title(f"Hidden units: {hidden}, Loss: {losses[-1]:.4f}")
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

axes[-1].axis("off")
plt.tight_layout()
plt.show()